In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk
!java -version
!javac -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.21.0.3+1-1~24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
[0.000s][warning][os,container] Cgroup memory controller path 

In [ ]:
%%writefile CampusSmartParking.java

import java.util.*;
import java.time.*;

// TASK 1: Vehicle class and object modelling
abstract class Vehicle {
    private final String regNo;

    Vehicle(String regNo) {
        if (regNo == null || regNo.isBlank())
            throw new IllegalArgumentException("Invalid registration");

        this.regNo = regNo;
    }

    public String getRegNo() {
        return regNo;
    }

    abstract double calculateFee(int minutes);

    @Override
    public boolean equals(Object obj) {
        return obj instanceof Vehicle &&
            regNo.equals(((Vehicle)obj).regNo);
    }

    @Override
    public int hashCode() {
        return Objects.hash(regNo);
    }

    @Override
    public String toString() {
        return getClass().getSimpleName() + ": " + regNo;
    }
}

// TASK 6 and 7: Inheritance and overriding
class TwoWheeler extends Vehicle {
    TwoWheeler(String regNo) {
        super(regNo);
    }

    @Override
    double calculateFee(int minutes) {
        return (minutes + 59) / 60 * 10;
    }
}

class Car extends Vehicle {
    Car(String regNo) {
        super(regNo);
    }

    @Override
    double calculateFee(int minutes) {
        return (minutes + 59) / 60 * 30;
    }
}

// TASK 9: Interface
interface Chargeable {
    double chargingFee(int units);
}

class ElectricCar extends Car implements Chargeable {
    ElectricCar(String regNo) {
        super(regNo);
    }

    @Override
    public double chargingFee(int units) {
        return units * 12;
    }
}

// TASK 12: New vehicle categories
class EBike extends Vehicle {
    EBike(String regNo) {
        super(regNo);
    }

    @Override
    double calculateFee(int minutes) {
        return 20;
    }
}

class DeliveryVan extends Vehicle {
    DeliveryVan(String regNo) {
        super(regNo);
    }

    double calculateFee(int minutes, LocalTime entry) {
        int hours = (minutes + 59) / 60;
        LocalTime exit = entry.plusMinutes(minutes);

        boolean peak = entry.isBefore(LocalTime.of(17, 0))
            && exit.isAfter(LocalTime.of(9, 0));

        return hours * (peak ? 60 : 30);
    }

    @Override
    double calculateFee(int minutes) {
        return calculateFee(minutes, LocalTime.NOON);
    }
}

// TASK 4 and 5: Constructors and static members
class ParkingTicket {
    private static int counter = 1000;

    private final int ticketNo;
    private final Vehicle vehicle;
    private final int floor;
    private final int slot;
    private final long entryTime;

    ParkingTicket(Vehicle vehicle, int floor, int slot) {
        this.ticketNo = ++counter;
        this.vehicle = vehicle;
        this.floor = floor;
        this.slot = slot;
        this.entryTime = System.currentTimeMillis();
    }

    public int getTicketNo() {
        return ticketNo;
    }

    public Vehicle getVehicle() {
        return vehicle;
    }

    public int getFloor() {
        return floor;
    }

    public int getSlot() {
        return slot;
    }

    public long getEntryTime() {
        return entryTime;
    }
}

// TASK 12: Pass abstraction
interface ParkingPass {
    boolean isValid();
}

class OneDayPass implements ParkingPass {
    private final LocalDate expiry;

    OneDayPass() {
        expiry = LocalDate.now();
    }

    @Override
    public boolean isValid() {
        return !LocalDate.now().isAfter(expiry);
    }
}

// TASK 3, 10 and 11: Parking facility
class ParkingFacility {
    private final String[][] slots;

    private final Map<String, ParkingTicket> activeTickets =
        new HashMap<>();

    private double totalCollection = 0;

    ParkingFacility(int[] capacity) {
        slots = new String[capacity.length][];

        for (int i = 0; i < capacity.length; i++) {
            if (capacity[i] < 0)
                throw new IllegalArgumentException("Invalid capacity");

            slots[i] = new String[capacity[i]];
        }
    }

    public int[] allocate(String regNo) {
        for (int f = 0; f < slots.length; f++) {
            for (int s = 0; s < slots[f].length; s++) {
                if (slots[f][s] == null) {
                    slots[f][s] = regNo;
                    return new int[]{f, s};
                }
            }
        }
        return null;
    }

    public boolean release(int floor, int slot) {
        if (floor < 0 || floor >= slots.length)
            return false;

        if (slot < 0 || slot >= slots[floor].length)
            return false;

        if (slots[floor][slot] == null)
            return false;

        slots[floor][slot] = null;
        return true;
    }

    public ParkingTicket enter(Vehicle vehicle) {
        String regNo = vehicle.getRegNo();

        if (activeTickets.containsKey(regNo)) {
            System.out.println("Duplicate registration: " + regNo);
            return null;
        }

        int[] position = allocate(regNo);

        if (position == null) {
            System.out.println("Parking Full");
            return null;
        }

        ParkingTicket ticket = new ParkingTicket(
            vehicle, position[0], position[1]);

        activeTickets.put(regNo, ticket);

        System.out.println("Entry successful: Ticket "
            + ticket.getTicketNo());

        return ticket;
    }

    public double exit(
        String regNo,
        int minutes,
        boolean pass,
        int chargingUnits
    ) {
        if (minutes < 0 || chargingUnits < 0)
            throw new IllegalArgumentException("Invalid input");

        ParkingTicket ticket = activeTickets.get(regNo);

        if (ticket == null) {
            System.out.println("Vehicle not found");
            return -1;
        }

        Vehicle vehicle = ticket.getVehicle();

        double fee = pass ? 0 : vehicle.calculateFee(minutes);

        if (!pass && vehicle instanceof Chargeable) {
            fee += ((Chargeable)vehicle).chargingFee(chargingUnits);
        }

        release(ticket.getFloor(), ticket.getSlot());
        activeTickets.remove(regNo);

        totalCollection += fee;

        System.out.println("Exit: " + regNo);
        System.out.println("Fee: Rs " + fee);

        return fee;
    }

    public double getTotalCollection() {
        return totalCollection;
    }
}

// MAIN PROGRAM
public class CampusSmartParking {

    public static void main(String[] args) {

        System.out.println("CAMPUS SMART PARKING SYSTEM");
        System.out.println("===========================");

        // TASK 2: Jagged array
        int[] capacity = {2, 3, 2};

        ParkingFacility parking =
            new ParkingFacility(capacity);

        // TASK 8: Runtime polymorphism
        Vehicle[] vehicles = {
            new Car("TN01AB1111"),
            new TwoWheeler("TN02CD2222"),
            new ElectricCar("TN03EF3333")
        };

        ParkingTicket[] tickets = new ParkingTicket[3];

        System.out.println("\nVEHICLE ENTRY");

        for (int i = 0; i < vehicles.length; i++) {
            tickets[i] = parking.enter(vehicles[i]);
        }

        System.out.println("\nVEHICLE EXIT");

        if (tickets[0] != null)
            parking.exit("TN01AB1111", 120, false, 0);

        if (tickets[1] != null)
            parking.exit("TN02CD2222", 60, false, 0);

        if (tickets[2] != null)
            parking.exit("TN03EF3333", 120, false, 10);

        System.out.println("\nTOTAL COLLECTION");
        System.out.println("Rs " + parking.getTotalCollection());

        // TASK 12: New vehicle categories
        System.out.println("\nNEW VEHICLE CATEGORIES");

        Vehicle ebike = new EBike("TN04EB4444");
        Vehicle van = new DeliveryVan("TN05DV5555");

        System.out.println("EBike Fee: Rs " +
            ebike.calculateFee(120));

        System.out.println("Delivery Van Fee: Rs " +
            ((DeliveryVan)van).calculateFee(
                120, LocalTime.of(10, 0)));

        // One-day pass
        ParkingPass pass = new OneDayPass();

        System.out.println("One Day Pass Valid: " +
            pass.isValid());

        // TASK 7: equals()
        Vehicle c1 = new Car("TN01AB1111");
        Vehicle c2 = new Car("TN01AB1111");

        System.out.println("\nEQUALS TEST");
        System.out.println(c1.equals(c2));
    }
}

Overwriting CampusSmartParking.java


In [ ]:
!javac CampusSmartParking.java
!java CampusSmartParking

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
CAMPUS SMART PARKING SYSTEM

VEHICLE ENTRY
Entry successful: Ticket 1001
Entry successful: Ticket 1002
Entry successful: Ticket 1003

VEHICLE EXIT
Exit: TN01AB1111
Fee: Rs 60.0
Exit: TN02CD2222
Fee: Rs 10.0
Exit: TN03EF3333
Fee: Rs 180.0

TOTAL COLLECTION
Rs 250.0

NEW VEHICLE CATEGORIES
EBike Fee: Rs 20.0
Delivery Van Fee: Rs 120.0
One Day